# 1. Ingest your Qualtrics export

**What this notebook does, in plain terms:**

1. Reads a raw Qualtrics CSV (the kind with three stacked header rows).
2. Writes a **codebook** (`output/codebook.csv`) listing every question number next to its question text, so you can see what `Q3` actually asked.
3. Writes a **config file** (`output/config.json`) with friendly names already filled in. You edit this once; the other notebooks read it.
4. Writes a **clean data file** (`output/responses_clean.csv`) used by the other two notebooks.

You do not need to know pandas. Run each cell top to bottom (Shift+Enter). The only thing you may want to edit by hand is `output/config.json`, and the codebook tells you what each number means.


In [ ]:
# --- standard setup (works on any OS; uses relative paths only) ---
from pathlib import Path
import json, re
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":          # launched from inside notebooks/
    ROOT = ROOT.parent
DATA_DIR = ROOT / "sample_data"       # put your own Qualtrics CSV here
OUT_DIR  = ROOT / "output"            # everything this tool writes lands here
OUT_DIR.mkdir(exist_ok=True)
print("Repo root :", ROOT)
print("Data dir  :", DATA_DIR)
print("Output dir:", OUT_DIR)

## Pick the file to read

By default this grabs the most recent `.csv` in `sample_data/`. To use your own data, drop your Qualtrics export into `sample_data/` and re-run, **or** set `RAW_FILE` to a path yourself.

In [ ]:
RAW_FILE = None   # e.g. RAW_FILE = DATA_DIR / "my_export.csv"

if RAW_FILE is None:
    csvs = sorted(DATA_DIR.glob("*.csv"), key=lambda p: p.stat().st_mtime)
    assert csvs, f"No CSV found in {DATA_DIR}. Put your Qualtrics export there."
    RAW_FILE = csvs[-1]
RAW_FILE = Path(RAW_FILE)
print("Reading:", RAW_FILE.name)

## Strip the Qualtrics header rows

A Qualtrics CSV stacks three rows at the top: (1) short question codes that become our column names, (2) the full question text, (3) an internal JSON row. We keep row 1 as columns, save row 2 as the codebook, and detect/skip the JSON row automatically (some exports omit it).

In [ ]:
raw = pd.read_csv(RAW_FILE, dtype=str, keep_default_na=False)
codes = list(raw.columns)
question_text = raw.iloc[0].tolist()      # second physical row = question text

def _looks_like_importid(row):
    return any(isinstance(v, str) and "ImportId" in v for v in row)

# Skip the import-id JSON row if present (3-row export); else 2-row export.
first_data_row = 2 if (len(raw) > 1 and _looks_like_importid(raw.iloc[1])) else 1
data = raw.iloc[first_data_row:].reset_index(drop=True)
print(f"{len(data)} response rows, {len(codes)} columns")

## Build the codebook

This is the piece that makes mapping-by-number usable: open `output/codebook.csv` and you can read what every `Q##` asked.

In [ ]:
codebook = pd.DataFrame({"code": codes, "question_text": question_text})
codebook.to_csv(OUT_DIR / "codebook.csv", index=False)
print("Wrote", OUT_DIR / "codebook.csv")
codebook

## Figure out column roles automatically

Qualtrics spreads one logical question across several columns. We sort them into metadata, timing (`..._Page Submit`), text-entry (`..._TEXT`), matrix items (`Q3_1`, `Q3_2`, ... sharing base `Q3`), and plain single questions. Matrix grids are recorded so the quality check can look for straight-lining. You can correct the groups in the config later.

In [ ]:
META = {
    "StartDate", "EndDate", "Status", "IPAddress", "Progress",
    "Duration (in seconds)", "Finished", "RecordedDate", "ResponseId",
    "RecipientLastName", "RecipientFirstName", "RecipientEmail",
    "ExternalReference", "LocationLatitude", "LocationLongitude",
    "DistributionChannel", "UserLanguage", "Q_RecaptchaScore",
}
TIMING_SUFFIXES = ("_Page Submit", "_First Click", "_Last Click", "_Click Count")

def role(code):
    if code in META:
        return "meta"
    if code.endswith(TIMING_SUFFIXES):
        return "timing"
    if code.endswith("_TEXT"):
        return "text"
    return "question"

def to_base(code):
    """Collapse a column to its logical question: Q3_1 -> Q3, Q5_10_TEXT -> Q5."""
    c = re.sub(r"_TEXT$", "", code)
    c = re.sub(r"_\d+$", "", c)
    return c

roles = {c: role(c) for c in codes}

from collections import OrderedDict
groups = OrderedDict()
for c in codes:
    if roles[c] in ("question", "text"):
        groups.setdefault(to_base(c), []).append(c)

matrix_groups = OrderedDict()
for base, members in groups.items():
    items = [m for m in members if re.match(rf"^{re.escape(base)}_\d+$", m)]
    if len(items) >= 2:
        matrix_groups[base] = items

print("Detected", len(matrix_groups), "matrix grid(s):")
for b, items in matrix_groups.items():
    print(f"  {b}: {items}")

## Suggest friendly names and write the config

We turn each question's text into a short readable name (rename freely). The config maps `friendly_name -> question number`. That is the point: you reference `comfort` or `age`, not a long verbatim label and not a raw code you had to memorize.

`config.json` is written **only if it does not already exist**, so re-running never clobbers your edits. A fresh suggestion always goes to `config_autogenerated.json`.

In [ ]:
text_by_code = dict(zip(codes, question_text))

def slug(text):
    text = text.split(" - ")[0]                       # drop matrix sub-item label
    words = re.sub(r"[^a-z0-9]+", " ", text.lower()).split()
    return "_".join(words[:6]) or "question"

questions, seen = OrderedDict(), {}
for base, members in groups.items():
    name = slug(text_by_code.get(base, text_by_code.get(members[0], base)))
    if name in seen:
        seen[name] += 1
        name = f"{name}_{seen[name]}"
    else:
        seen[name] = 1
    questions[name] = base

config = {
    "_README": "Map friendly names to Qualtrics question numbers. Open "
               "output/codebook.csv to see what each number asked. Edit "
               "'questions', then run notebooks 2 and 3. Fill 'attention_checks' "
               "with {question_code: expected_answer} if your survey has any.",
    "questions": questions,
    "straightline_groups": matrix_groups,
    "attention_checks": {},
    "quality": {
        "total_duration_floor_sec": 180,
        "page_floor_sec": 2.0,
        "min_fast_pages_to_flag": 2,
        "recaptcha_min": 0.5,
        "attention_fail_min": 1,
        "min_items_for_straightline": 5,
        "min_straightline_matrices_to_flag": 1,
        "exclude_min_flags": 2,
        "count_incomplete_toward_exclusion": False,
    },
}

(OUT_DIR / "config_autogenerated.json").write_text(json.dumps(config, indent=2))
cfg_path = OUT_DIR / "config.json"
if not cfg_path.exists():
    cfg_path.write_text(json.dumps(config, indent=2))
    print("Wrote a starter", cfg_path.name, "- open it and edit the friendly names.")
else:
    print(cfg_path.name, "already exists; left it untouched.")
    print("A fresh suggestion is in config_autogenerated.json.")
print("\nSuggested names -> question number:")
for k, v in questions.items():
    print(f"  {k:<28} {v}")

## Write the clean data file

One header row of codes, just the responses. By default we drop direct identifiers (IP, GPS, recipient name/email) so the working file is safer to share. Set `DEIDENTIFY = False` to keep them.

In [ ]:
DEIDENTIFY = True
DROP = {"IPAddress", "LocationLatitude", "LocationLongitude",
        "RecipientLastName", "RecipientFirstName", "RecipientEmail",
        "ExternalReference"}

clean = data.drop(columns=[c for c in DROP if c in data.columns]) if DEIDENTIFY else data.copy()
clean.to_csv(OUT_DIR / "responses_clean.csv", index=False)
print("Wrote", OUT_DIR / "responses_clean.csv", f"({len(clean)} rows, {clean.shape[1]} cols)")
print("Done. Next: open output/config.json, then run 02_quality_check.ipynb.")